# StART — Deep Learning Model Review (Jupyter / VS Code)

Real laptop-safe PyTorch model → evidence pipeline (EV-DL-0001..0007) →
dual-mode agent review → figures → proof-carrying report.

Select the kernel **Python (StART .venv-start)**. The LLM (if enabled)
reasons only over the evidence bundle and never sees raw data; the
default mode is deterministic and needs no API key.


## 1. Review options

Edit these directly, or use the interactive widgets in the next cell if
`ipywidgets` is installed. Architecture options: `mlp`, `leaky_relu_mlp`,
`residual_mlp`, `wide_deep`. Agent mode: `deterministic` or `llm`.


In [ ]:
OPTIONS = {
    "architecture": "mlp",
    "epochs": 8,            # laptop-safe: <= 10
    "batch_size": 128,      # laptop-safe: <= 128
    "learning_rate": 0.001,
    "explain_method": "integrated_gradients",  # or "gradient_shap"
    "sensitivity_cohort": "test",              # test | oos | development
    "agent_mode": "deterministic",             # deterministic | llm
    "llm_provider": "none",                    # none | openai | anthropic | enterprise_llm_gateway
}
print(OPTIONS)

## 2. Optional interactive widgets

Run this cell to adjust options with dropdowns/sliders. If `ipywidgets`
is unavailable, skip it — the dictionary above is used as-is.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    arch = widgets.Dropdown(options=['mlp','leaky_relu_mlp','residual_mlp','wide_deep'],
                            value=OPTIONS['architecture'], description='architecture')
    epochs = widgets.IntSlider(value=OPTIONS['epochs'], min=1, max=10, description='epochs')
    agent = widgets.Dropdown(options=['deterministic','llm'],
                             value=OPTIONS['agent_mode'], description='agent_mode')
    provider = widgets.Dropdown(options=['none','openai','anthropic','enterprise_llm_gateway'],
                                value=OPTIONS['llm_provider'], description='llm_provider')
    cohort = widgets.Dropdown(options=['test','oos','development'],
                              value=OPTIONS['sensitivity_cohort'], description='cohort')
    display(arch, epochs, agent, provider, cohort)
    _WIDGETS = (arch, epochs, agent, provider, cohort)
except ImportError:
    _WIDGETS = None
    print('ipywidgets not installed; using the OPTIONS dict above.')

## 3. Environment, device, and key handling

Default deterministic mode needs no key. In LLM mode, the key is read
from the environment or a hidden `getpass` prompt — session-only, never
persisted or echoed. On Databricks use a secret scope instead.


In [ ]:
from start.modeling.deep_learning import captum_available, resolve_torch_device, torch_available

if _WIDGETS is not None:
    arch, epochs, agent, provider, cohort = _WIDGETS
    OPTIONS.update(architecture=arch.value, epochs=epochs.value, agent_mode=agent.value,
                   llm_provider=provider.value, sensitivity_cohort=cohort.value)

assert torch_available(), 'Install the torch extra: pip install -e ".[torch]"'
print('device (CUDA -> MPS -> CPU):', resolve_torch_device())
print('captum available:', captum_available())

if OPTIONS['agent_mode'] == 'llm' and OPTIONS['llm_provider'] not in ('none','enterprise_llm_gateway'):
    from start.providers.keys import ensure_provider_key
    status = ensure_provider_key(OPTIONS['llm_provider'], prompt_for_key=None)
    print(f"LLM key source: {status.source}")  # source only, never the key
    if not status.ok:
        print('WARNING: no key; falling back to deterministic mode explicitly.')
        OPTIONS['agent_mode'] = 'deterministic'; OPTIONS['llm_provider'] = 'none'

## 4. Run the full review

Train → evidence (EV-DL-0001..0007) → agent review → figures → report,
all through the reusable `run_dl_review` workflow.


In [ ]:
from start.modeling.dl_training import DLReviewOptions, run_dl_review

opts = DLReviewOptions(
    architecture=OPTIONS['architecture'],
    epochs=min(OPTIONS['epochs'], 10),
    batch_size=min(OPTIONS['batch_size'], 128),
    learning_rate=OPTIONS['learning_rate'],
    explain_method=OPTIONS['explain_method'],
    sensitivity_cohort=OPTIONS['sensitivity_cohort'],
    agent_mode=OPTIONS['agent_mode'],
    llm_provider='' if OPTIONS['llm_provider'] == 'none' else OPTIONS['llm_provider'],
)
result = run_dl_review(opts)

## 5. Evidence, findings, and sign-off


In [ ]:
import pandas as pd

rows = [{'evidence_id': r.artifacts.get('dl_evidence_label', r.evidence_id),
         'diagnostic': r.test_name, 'status': r.status.value} for r in result.evidence]
display(pd.DataFrame(rows))

ar = result.agent_review
print('Agent mode:', 'llm-assisted (' + ar.llm_provider + ')' if ar.mode == 'llm' else 'deterministic')
print('Evidence critique status:', 'PASSED' if ar.critique_ok else 'FAILED')
for title, items in (('Challenger memo', ar.challenge_memo),
                     ('Governance assessment', ar.governance)):
    print(f'\n## {title}')
    for item in items:
        print('-', item)
print('\n## Sign-off recommendation\n' + ar.signoff)

## 6. Cohort metrics


In [ ]:
import pandas as pd
display(pd.DataFrame(result.cohort_metrics).T)

## 7. Figures (inline)


In [ ]:
from IPython.display import Image, display
for name, path in sorted(result.figures.items()):
    print(name)
    display(Image(filename=path))

## 8. Report path


In [ ]:
print('Report:', result.report_path)